# Taller Final - Corte 2: Reingeniería de Software (30%)
**Estudiante:** Juan Caballero  
**Institución:** Universidad de La Sabana  
**Profesor:** Diego Zuluaga  

---
### Descripción del Proyecto
Este sistema representa una reingeniería completa del MVP inicial, migrando de una lógica simple a una arquitectura profesional basada en:
1. **POO:** Encapsulamiento, Herencia y Polimorfismo.
2. **Persistencia:** Base de datos SQLite normalizada (3NF).
3. **Robustez:** Manejo de excepciones (Try-Except) y validaciones.

In [1]:
import sqlite3

# --- 1. CAPA DE MODELO (POO) ---
class EntidadBase:
    """Clase padre para aplicar Herencia"""
    def __init__(self, nombre):
        self._nombre = nombre # Atributo protegido (Encapsulamiento)

class Producto(EntidadBase):
    def __init__(self, nombre, precio, stock, nit_proveedor):
        super().__init__(nombre)
        self.__precio = precio # Atributo privado
        self.stock = stock
        self.nit_proveedor = nit_proveedor

    def get_precio(self):
        return self.__precio

# --- 2. CAPA DE PERSISTENCIA (SQL) ---
class SistemaCafeteria:
    def __init__(self):
        try:
            self.conexion = sqlite3.connect("cafeteria_profesional.db")
            self.conexion.execute("PRAGMA foreign_keys = ON;")
            self.cursor = self.conexion.cursor()
            self._crear_tablas()
        except sqlite3.Error as e:
            print(f"❌ Error crítico al conectar la BD: {e}")

    def _crear_tablas(self):
        # Tablas normalizadas (1NF, 2NF, 3NF)
        self.cursor.execute('''CREATE TABLE IF NOT EXISTS proveedores (
            nit TEXT PRIMARY KEY,
            nombre_empresa TEXT NOT NULL,
            ciudad TEXT)''')
        
        self.cursor.execute('''CREATE TABLE IF NOT EXISTS productos (
            id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre TEXT NOT NULL,
            precio REAL NOT NULL,
            stock INTEGER NOT NULL,
            nit_proveedor TEXT,
            FOREIGN KEY (nit_proveedor) REFERENCES proveedores(nit))''')
        self.conexion.commit()

    # --- 3. MANEJO DE ERRORES Y CRUD ---
    def registrar_producto(self, producto):
        try:
            # Validación de datos antes de procesar
            if producto.get_precio() <= 0:
                raise ValueError("El precio debe ser un número positivo.")
            
            self.cursor.execute('''INSERT INTO productos (nombre, precio, stock, nit_proveedor) 
                                   VALUES (?, ?, ?, ?)''', 
                                (producto._nombre, producto.get_precio(), producto.stock, producto.nit_proveedor))
            self.conexion.commit()
            print(f"✅ Producto '{producto._nombre}' registrado con éxito.")
        except ValueError as ve:
            print(f"⚠️ Error de validación: {ve}")
        except sqlite3.IntegrityError:
            print("❌ Error: El proveedor no existe en la base de datos.")
        except Exception as e:
            print(f"🔥 Error inesperado: {e}")

    # --- 4. CONSULTA AVANZADA (JOIN) ---
    def generar_reporte_inventario(self):
        print("\n--- REPORTE DE INVENTARIO Y PROVEEDORES ---")
        query = '''
        SELECT p.nombre, p.precio, p.stock, pr.nombre_empresa
        FROM productos p
        INNER JOIN proveedores pr ON p.nit_proveedor = pr.nit
        ORDER BY p.stock ASC
        '''
        self.cursor.execute(query)
        for p, pre, s, prov in self.cursor.fetchall():
            print(f"📦 {p} | Precio: ${pre} | Stock: {s} | Prov: {prov}")

# --- 5. PRUEBAS DEL SISTEMA (EJECUCIÓN) ---
if __name__ == "__main__":
    app = SistemaCafeteria()
    
    # Insertar proveedor base
    app.cursor.execute("INSERT OR IGNORE INTO proveedores VALUES ('9001', 'Sabana Gourmet', 'Chía')")
    
    # Intentar registrar productos (con y sin errores)
    p1 = Producto("Capuccino", 6500, 20, "9001")
    p2 = Producto("Error Test", -100, 5, "9001") # Esto disparará el ValueError
    
    app.registrar_producto(p1)
    app.registrar_producto(p2)
    
    app.generar_reporte_inventario()
    app.conexion.close()

✅ Producto 'Capuccino' registrado con éxito.
⚠️ Error de validación: El precio debe ser un número positivo.

--- REPORTE DE INVENTARIO Y PROVEEDORES ---
📦 Capuccino | Precio: $6500.0 | Stock: 20 | Prov: Sabana Gourmet
📦 Capuccino | Precio: $6500.0 | Stock: 20 | Prov: Sabana Gourmet
📦 Capuccino | Precio: $6500.0 | Stock: 20 | Prov: Sabana Gourmet
📦 Capuccino | Precio: $6500.0 | Stock: 20 | Prov: Sabana Gourmet
